# ORB 기반 비디오 객체 추적 실습

ORB 특징점과 BF 매처를 활용하여 비디오에서 타겟 영역을 프레임 단위로 추적하고, 검출/매칭 속도 및 FPS를 측정하는 실습

In [ ]:
import cv2
import numpy as np
import time
import urllib.request
from google.colab.patches import cv2_imshow

## 샘플 영상 로드 및 타겟 설정

In [ ]:
# 샘플 영상 로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi'
urllib.request.urlretrieve(url, 'vtest.avi')
cap = cv2.VideoCapture('vtest.avi')

# 타겟 설정 (첫 프레임의 특정 영역)
ret, first_frame = cap.read()
if not ret:
    exit()

target_gray = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)
target_roi = target_gray[100:300, 200:400]  # 보행자가 있는 영역 절삭

## ORB 및 매처 초기화

In [ ]:
# ORB 및 매처 초기화
orb = cv2.ORB_create(nfeatures=1000)
kp1, des1 = orb.detectAndCompute(target_roi, None)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

## 프레임별 추적 및 속도 측정

In [ ]:
print(f"{'Frame':<8} | {'Detect(ms)':<12} | {'Match(ms)':<12} | {'Total(ms)':<12} | {'FPS':<6}")
print("-" * 65)

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # --- 속도 측정 시작 ---
    start_total = time.time()

    # 특징점 검출 및 기술자 생성 시간 측정
    start_det = time.time()
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    kp2, des2 = orb.detectAndCompute(gray_frame, None)
    end_det = time.time()
    det_time = (end_det - start_det) * 1000  # ms 단위

    match_time = 0
    good_matches = []

    if des2 is not None:
        # 매칭 시간 측정
        start_match = time.time()
        matches = bf.knnMatch(des1, des2, k=2)
        good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
        end_match = time.time()
        match_time = (end_match - start_match) * 1000

    # --- 속도 측정 종료 ---
    end_total = time.time()
    total_time = (end_total - start_total) * 1000
    fps = 1.0 / (end_total - start_total) if total_time > 0 else 0

    # 지표 출력 (매 10프레임마다)
    if frame_count % 10 == 0:
        print(f"{frame_count:<8} | {det_time:<12.2f} | {match_time:<12.2f} | {total_time:<12.2f} | {fps:<6.1f}")

    # 시각화 (매 100프레임마다)
    if frame_count % 100 == 0:
        res = cv2.drawMatches(target_roi, kp1, frame, kp2,
                              good_matches[:20], None,
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        cv2.putText(res, f"FPS: {fps:.1f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2_imshow(res)

    frame_count += 1

cap.release()
print(f"\nTotal frames processed: {frame_count}")